In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2
# sys.path.append('/home/wolfgang/repos/sleep_research_io')
# from sleep_research_functions import *
%matplotlib widget
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from icu_sleep_breathing_2020_help_functions import * 

pd.options.display.max_rows = 300
pd.options.display.max_columns = 300

font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 7}

font_headers =  {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 8}

matplotlib.rc('font', **font)

In [2]:
plots_savedir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/plots'

[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3254: DtypeWarning: Columns (15,91,92,93,6455,6456,6457,12819,12820,12821,19185,19186,19187,21491,21492,21493,21519,21520,21521,21526,21527,21528,25549,25550,25551,31913,31914,31915,38279,38280,38281,44643,44644,44645,46851,46852,46853,46865,46866,46867,46879,46880,46881,46949,46950,46951,46956,46957,46958,46963,46964,46965,46970,46971,46972,46977,46978,46979,46984,46985,46986,51007,51008,51009,53336,53337,53338,57305,57373,57374,57375,59588,59589,59590,59602,59603,59604,59686,59687,59688,63669,63737,63738,63739,65952,65953,65954,65959,65960,65961,65980,65981,65982,66036,66037,66038,66043,66044,66045,66050,66051,66052,66057,66058,66059,66071,66072,66073,66078,66079,66080,70033,70101,70102,70103,72318,72319,72320,72332,72333,72334,72402,72403,72404,72416,72417,72418,76399,76467,76468,76469,78682,78683,78684,78696,78697,78698,78710,78711,78712,78766,78767,78768,78787,78788,78789,78801,78

# of subjects ICU before inclusion criteria: 102
# of 12-hour segments ICU before inclusion criteria: 1161
# of subjects ICU after inclusion criteria: 102
# of 12-hour segments ICU after inclusion criteria: 1161
24-hour segments: 387
12-hour segments: 774

# of subjects sleeplab before inclusion criteria: 412
# of 12-hour segments sleeplab before inclusion criteria: 412


In [3]:
for modality in ['breathing', 'ecg_nn']:
    var_n2 = 'stages_distribution_MODALITY_agrelaxed_N2'.replace('MODALITY', modality)
    var_n3 = 'stages_distribution_MODALITY_agrelaxed_N3'.replace('MODALITY', modality)
    var_sum = 'stages_distribution_MODALITY_agrelaxed_N2N3'.replace('MODALITY', modality)
    
    summary_days_icu[var_sum] = summary_days_icu[var_n2] + summary_days_icu[var_n3]
    summary_days_sleeplab[var_sum] = summary_days_sleeplab[var_n2] + summary_days_sleeplab[var_n3]

In [4]:
for agreement in ['agreement_relaxed', 'agreement']:
    for modality in ['amendsumeffort', 'ecg_nn']:
        var_sleep = 'hours_sleep_MODALITY_all'.replace('MODALITY', modality) # all overlap sleep
        var_agreeing = 'hours_sleep_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        
        if agreement == 'agreement_relaxed':
             # all overlap sleep
            var_discordant = 'hours_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement) # discordant sleep among overlap sleep
        elif agreement == 'agreement':
            var_discordant = 'hours_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)

            
        summary_days_icu[var_discordant] = summary_days_icu[var_sleep] - summary_days_icu[var_agreeing]
        summary_days_sleeplab[var_discordant] = summary_days_sleeplab[var_sleep] - summary_days_sleeplab[var_agreeing]

In [5]:
summary_days_icu.loc[:, ['stages_distribution' in x for x in summary_days_icu.columns]] *= 100
summary_days_sleeplab.loc[:, ['stages_distribution' in x for x in summary_days_sleeplab.columns]] *= 100

summary_f_icu = summary_days_icu.loc[summary_days_icu.day_cat == 'f', :]
summary_dn_icu = summary_days_icu.loc[summary_days_icu.day_cat != 'f', :]

summary_days_sleeplab_full = summary_days_sleeplab.copy()

In [6]:
### 3x2 plot
# cols: data type, rows: stage version (resp, ecg, expert)

variables_template = ['sleep_hours_MODALITY',
             'stages_distribution_MODALITY_S',
             'stages_distribution_MODALITY_R',
             'stages_distribution_MODALITY_N1',
            'stages_distribution_MODALITY_N2',
             'stages_distribution_MODALITY_N3',
             'sleep_MODALITY_sfi',
             'sleep_MODALITY_sfi_w',
             'sleep_MODALITY_arousali',
            ]
labels = ['Sleep Dur. (h)', 'Sleep Efficiency', 'Stage R (%)', 'Stage N1 (%)', 'Stage N2 (%)', 'Stage N3 (%)', 'SFI', 'SFI Wake', 'Arousal I.']

In [25]:
Xy_sleeplab = summary_days_sleeplab_full[summary_days_sleeplab_full['matched_all'] == 1].copy()

days = 'full'
agreement = 'agreement_relaxed'

# min_hours_sleep_overlap_icu = 2
# if days == 'day_night':
#     Xy_icu = summary_dn_icu.loc[np.any([summary_dn_icu.hours_sleep_ecg_nn_all >= min_hours_sleep_overlap_icu],
#                                                    axis=0), :].copy()
# elif days == 'full':
#     Xy_icu = summary_f_icu.loc[np.any([summary_f_icu.hours_sleep_ecg_nn_all >= min_hours_sleep_overlap_icu],
#                                                    axis=0), :].copy()
    
min_hours_sleep_icu = 2
if days == 'day_night':
    Xy_icu = summary_dn_icu.loc[np.any([summary_dn_icu[f'hours_sleep_amendsumeffort_{agreement}'] >= min_hours_sleep_icu, 
                                                   summary_dn_icu[f'hours_sleep_ecg_nn_{agreement}'] >= min_hours_sleep_icu],
                                                   axis=0), :].copy()
elif days == 'full':
    Xy_icu = summary_f_icu.loc[np.any([summary_f_icu[f'hours_sleep_amendsumeffort_{agreement}'] >= min_hours_sleep_icu, 
                                                   summary_f_icu[f'hours_sleep_ecg_nn_{agreement}'] >= min_hours_sleep_icu],
                                                   axis=0), :].copy()

print(f'Segments: {Xy_icu.shape[0]}')
print(f'Subjects: {Xy_icu.study_id.unique().shape[0]}')

Segments: 163
Subjects: 80


In [26]:
sleep_total =  f'hours_sleep_{modality}_all'
sleep_agree = f'hours_sleep_{modality}_{agreement}'
sleep_discordant = f'hours_discordant_{modality}_{agreement}'
discordant_proportion = f'discordant_proportion_{modality}_{agreement}'

In [27]:
Xy_icu[discordant_proportion] = np.divide(Xy_icu[sleep_discordant], Xy_icu[sleep_total])

In [28]:
Xy_icu[[sleep_agree, sleep_discordant, sleep_total, discordant_proportion]].head()

,hours_sleep_amendsumeffort_agreement_relaxed,hours_discordant_amendsumeffort_agreement_relaxed,hours_sleep_amendsumeffort_all,discordant_proportion_amendsumeffort_agreement_relaxed
5,3.875001,1.083333,4.958334,0.218487
11,2.908334,4.716667,7.625001,0.618579
17,5.408334,0.491667,5.900001,0.083333
26,3.808334,0.325000,4.133334,0.078629
29,2.125001,6.791667,8.916668,0.761682


In [29]:
[x for x in Xy_icu.columns if 'discordant' in x]

['hours_discordant_amendsumeffort_agreement_relaxed',
 'hours_discordant_ecg_nn_agreement_relaxed',
 'hours_discordant_amendsumeffort_agreement',
 'hours_discordant_ecg_nn_agreement',
 'discordant_proportion_amendsumeffort_agreement_relaxed']

In [30]:
from scipy.stats import ttest_1samp, pearsonr, spearmanr, ttest_rel, wilcoxon
import itertools

### compute table with feature value for each sleep stage and agreeing | discordant sleep

In [32]:
variables = ['hrv_nn',  'hrv_vlf', 'hrv_lf', 'hrv_hf', 'hrv_rmssd',
            'ibi', 'rr_10s_smooth', 'instability_30sec', 'ventilation_cvar_30sec',
            'cpc_lfc', 'cpc_hfc',
            ]

# statistic = 'mean'

In [33]:
cols = list(itertools.product(['Breathing', 'HRV'], ['N_Segments', 'Agree', 'Discordant', '(wil-stat, pval)', 'EffectPositive'])) + ['Significance', 'Sig_Breathing', 'Sig_HRV'] # , '(t-stat, pval)'
index = list(itertools.product(['N1', 'N2', 'N3', 'R'], variables))
df_fvalues = pd.DataFrame(columns= cols, index=index)

In [34]:
stage = 'N1'
modality = 'ecg_nn'
variable = 'cpc_hfc_lfc_ratio'
statistic_extracted = 'mean'

for stage in ['N1', 'N2', 'N3', 'R']:
    for variable in variables:
        for modality in ['ecg_nn', 'amendsumeffort']:

            j_col = modality.replace('ecg_nn', 'HRV').replace('amendsumeffort', 'Breathing')
            j_index = (stage, variable)

            feature_agree = variable + '_stagewise_agrelaxed_' + modality + '_' + stage + '_' + statistic_extracted
            feature_discordant = variable + '_stagewise_disrelaxed_' + modality + '_' + stage + '_' + statistic_extracted

            if variable in ['cpc_lfc', 'cpc_hfc']:
                # For HRV derived measures, the code performs some quality check that i did not do for CPC calculation. so it's weird to use those
                # CPC results if the HRV did not qualify. so here, let's remove those CPC entries. 
                # basically same code as default except we further also only take the rows with good HRV.
                hrv_good_quality = Xy_icu[[feature_agree.replace(variable, 'hrv_lf'), feature_discordant.replace(variable, 'hrv_lf')]]
                hrv_good_quality = np.all(pd.notna(hrv_good_quality), axis=1).values
                values = Xy_icu[[feature_agree, feature_discordant]].replace('Series([], Name: hrv_nn, dtype: float32)', np.nan).astype(float)
                values = values.iloc[hrv_good_quality, :]
                values = values.dropna(axis=0, how='any').copy()

            else:
                values = Xy_icu[[feature_agree, feature_discordant]].replace('Series([], Name: hrv_nn, dtype: float32)', np.nan).astype(float).dropna(axis=0, how='any').copy()
            
            df_fvalues.loc[j_index, (j_col, 'N_Segments')] = values.shape[0]

            values_agree = values[feature_agree].values
            values_discordant = values[feature_discordant].values

            decimals=1
            if np.any([np.abs(np.mean(values_agree)) < 1, np.abs(np.mean(values_discordant)) < 1]):
                decimals=3
            elif np.any([np.abs(np.mean(values_agree)) < 1, np.abs(np.mean(values_discordant)) < 10]):
                decimals=2
            df_fvalues.loc[j_index, (j_col, 'Agree')] = np.mean(values_agree).round(decimals)
            df_fvalues.loc[j_index, (j_col, 'Discordant')] = np.mean(values_discordant).round(decimals)
            df_fvalues.loc[j_index, (j_col, 'EffectPositive')] = np.mean(values_agree).round(decimals) < np.mean(values_discordant).round(decimals)
    #         df_fvalues.loc[j_index, (j_col, '(t-stat, pval)')] = ttest_rel(values_agree, values_discordant)
            W, p = wilcoxon(values_agree, values_discordant)
            df_fvalues.loc[j_index, (j_col, '(wil-stat, pval)')] = (int(W), np.round(p, 5))

        p_hrv = df_fvalues.loc[[j_index], [('HRV', '(wil-stat, pval)')]].values[0][0][1]
        p_breathing = df_fvalues.loc[[j_index], [('Breathing', '(wil-stat, pval)')]].values[0][0][1]

        df_fvalues.loc[j_index, 'Significance'] = (p_hrv <= 0.01) | (p_breathing <= 0.01)
        df_fvalues.loc[j_index, 'Sig_Breathing'] = p_breathing <= 0.01
        df_fvalues.loc[j_index, 'Sig_HRV'] = p_hrv <= 0.01

In [35]:
df_fvalues_save = df_fvalues[[('Breathing', 'N_Segments'),           ('Breathing', 'Agree'),       ('Breathing', 'Discordant'),
       ('Breathing', '(wil-stat, pval)'), ('HRV', 'N_Segments'),
                        ('HRV', 'Agree'),             ('HRV', 'Discordant'),
             ('HRV', '(wil-stat, pval)'),   
                          'Significance']]

df_fvalues_save.to_csv('Discordant_Sleep_FeatureDifference_Analysis_Details.csv')

In [36]:
df_fvalues_save

,"(Breathing, N_Segments)","(Breathing, Agree)","(Breathing, Discordant)","(Breathing, (wil-stat, pval))","(HRV, N_Segments)","(HRV, Agree)","(HRV, Discordant)","(HRV, (wil-stat, pval))",Significance
"(N1, hrv_nn)",100,0.756,0.76,"(2091, 0.13564)",71,0.791,0.792,"(1229, 0.77889)",False
"(N1, hrv_vlf)",80,1914.6,1474.2,"(840, 0.00018)",59,2297.9,2172.6,"(773, 0.3979)",True
"(N1, hrv_lf)",80,1129.3,914.3,"(916, 0.00073)",59,1376.2,1388.9,"(848, 0.78003)",True
"(N1, hrv_hf)",80,2055,1793.7,"(994, 0.00268)",59,2858.1,3010.8,"(847, 0.77425)",True
"(N1, hrv_rmssd)",80,65.5,60,"(926, 0.00087)",59,76.3,73.1,"(655, 0.08256)",True
"(N1, ibi)",104,3.98,4.01,"(2595, 0.66155)",71,3.95,3.55,"(518, 1e-05)",True
"(N1, rr_10s_smooth)",104,17.7,17.4,"(2246, 0.11653)",71,17.8,19.5,"(537, 2e-05)",True
"(N1, instability_30sec)",104,0.243,0.238,"(2286, 0.14992)",71,0.225,0.194,"(653, 0.00034)",True
"(N1, ventilation_cvar_30sec)",104,0.235,0.233,"(2446, 0.35707)",71,0.218,0.181,"(554, 3e-05)",True
"(N1, cpc_lfc)",80,4.29,3.38,"(978, 0.00208)",59,4.92,2.67,"(588, 0.02498)",True


In [37]:
print(df_fvalues.Significance.values.astype(int).shape)
print(df_fvalues.Significance.values.astype(int).sum())
print(df_fvalues.Significance.values.astype(int).mean().round(3))

(44,)
28
0.636


In [38]:
print('Any significance:')
print(list(df_fvalues.index[df_fvalues.Significance==True]))
print('')
print('Breathing Disc. sleep. Significance:')
print(list(df_fvalues.index[df_fvalues.Sig_Breathing==True]))
print('')
print('HRV Disc. sleep.')
print(list(df_fvalues.index[df_fvalues.Sig_HRV==True]))

Any significance:
[('N1', 'hrv_vlf'), ('N1', 'hrv_lf'), ('N1', 'hrv_hf'), ('N1', 'hrv_rmssd'), ('N1', 'ibi'), ('N1', 'rr_10s_smooth'), ('N1', 'instability_30sec'), ('N1', 'ventilation_cvar_30sec'), ('N1', 'cpc_lfc'), ('N1', 'cpc_hfc'), ('N2', 'hrv_nn'), ('N2', 'hrv_vlf'), ('N2', 'hrv_lf'), ('N2', 'hrv_hf'), ('N2', 'ibi'), ('N2', 'rr_10s_smooth'), ('N2', 'instability_30sec'), ('N2', 'ventilation_cvar_30sec'), ('N3', 'hrv_nn'), ('N3', 'hrv_vlf'), ('N3', 'hrv_lf'), ('N3', 'hrv_hf'), ('N3', 'hrv_rmssd'), ('N3', 'instability_30sec'), ('N3', 'ventilation_cvar_30sec'), ('N3', 'cpc_lfc'), ('R', 'hrv_rmssd'), ('R', 'cpc_hfc')]

Breathing Disc. sleep. Significance:
[('N1', 'hrv_vlf'), ('N1', 'hrv_lf'), ('N1', 'hrv_hf'), ('N1', 'hrv_rmssd'), ('N1', 'cpc_lfc'), ('N1', 'cpc_hfc'), ('N2', 'hrv_nn'), ('N2', 'hrv_vlf'), ('N2', 'hrv_lf'), ('N2', 'hrv_hf'), ('N2', 'ibi'), ('N2', 'rr_10s_smooth'), ('N2', 'instability_30sec'), ('N2', 'ventilation_cvar_30sec'), ('N3', 'hrv_vlf'), ('N3', 'hrv_lf'), ('N3', '

In [39]:
print('Breathing Disc. Sleep, Elevated Mean for Discordant Sleep:')
print(list(df_fvalues[(df_fvalues.Sig_Breathing==True) & (df_fvalues[('Breathing', 'EffectPositive')]==True)].index))
print('')
print('Breathing Disc. Sleep, Decreased Mean for Discordant Sleep:')
print(list(df_fvalues[(df_fvalues.Sig_Breathing==True) & (df_fvalues[('Breathing', 'EffectPositive')]==False)].index))
print('')
print('HRV Disc. Sleep, Elevated Mean for Discordant Sleep:')
print(list(df_fvalues[(df_fvalues.Sig_HRV==True) & (df_fvalues[('HRV', 'EffectPositive')]==True)].index))
print('')
print('HRV Disc. Sleep, Decreased Mean for Discordant Sleep:')
print(list(df_fvalues[(df_fvalues.Sig_HRV==True) & (df_fvalues[('HRV', 'EffectPositive')]==False)].index))

Breathing Disc. Sleep, Elevated Mean for Discordant Sleep:
[('N2', 'hrv_vlf'), ('N2', 'hrv_lf'), ('N2', 'rr_10s_smooth'), ('N3', 'hrv_vlf'), ('N3', 'hrv_lf'), ('N3', 'hrv_hf'), ('N3', 'hrv_rmssd'), ('N3', 'cpc_lfc'), ('R', 'hrv_rmssd'), ('R', 'cpc_hfc')]

Breathing Disc. Sleep, Decreased Mean for Discordant Sleep:
[('N1', 'hrv_vlf'), ('N1', 'hrv_lf'), ('N1', 'hrv_hf'), ('N1', 'hrv_rmssd'), ('N1', 'cpc_lfc'), ('N1', 'cpc_hfc'), ('N2', 'hrv_nn'), ('N2', 'hrv_hf'), ('N2', 'ibi'), ('N2', 'instability_30sec'), ('N2', 'ventilation_cvar_30sec')]

HRV Disc. Sleep, Elevated Mean for Discordant Sleep:
[('N1', 'rr_10s_smooth'), ('N2', 'rr_10s_smooth'), ('N3', 'instability_30sec'), ('N3', 'ventilation_cvar_30sec'), ('N3', 'cpc_lfc')]

HRV Disc. Sleep, Decreased Mean for Discordant Sleep:
[('N1', 'ibi'), ('N1', 'instability_30sec'), ('N1', 'ventilation_cvar_30sec'), ('N3', 'hrv_nn')]


In [40]:
df_effectdir = df_fvalues.loc[df_fvalues.Significance==True, ['Sig_Breathing', ('Breathing', 'EffectPositive'), 'Sig_HRV', ('HRV', 'EffectPositive')]].copy()

In [41]:
df_effectdir['Breathing_Effect_DiscordantSleep'] = np.nan
df_effectdir['HRV_Effect_DiscordantSleep'] = np.nan

for jloc, row in df_effectdir.iterrows():
    if row.Sig_Breathing == False:
        df_effectdir.loc[jloc, 'Breathing_Effect_DiscordantSleep'] = ''
    else:
        if row[('Breathing', 'EffectPositive')] == True:
            df_effectdir.loc[jloc, 'Breathing_Effect_DiscordantSleep'] = '+'
        else:
            df_effectdir.loc[jloc, 'Breathing_Effect_DiscordantSleep'] = '-'
            
    if row.Sig_HRV == False:
        df_effectdir.loc[jloc, 'HRV_Effect_DiscordantSleep'] = ''
    else:
        if row[('HRV', 'EffectPositive')] == True:
            df_effectdir.loc[jloc, 'HRV_Effect_DiscordantSleep'] = '+'
        else:
            df_effectdir.loc[jloc, 'HRV_Effect_DiscordantSleep'] = '-'


In [42]:
df_effectdir[['Breathing_Effect_DiscordantSleep', 'HRV_Effect_DiscordantSleep']].to_csv('Discordant_Sleep_FeatureDifference_Analysis_Overview.csv')

In [43]:
df_effectdir.head()

,Sig_Breathing,"(Breathing, EffectPositive)",Sig_HRV,"(HRV, EffectPositive)",Breathing_Effect_DiscordantSleep,HRV_Effect_DiscordantSleep
"(N1, hrv_vlf)",True,False,False,False,-,
"(N1, hrv_lf)",True,False,False,True,-,
"(N1, hrv_hf)",True,False,False,True,-,
"(N1, hrv_rmssd)",True,False,False,False,-,
"(N1, ibi)",False,True,True,False,,-


### compute correlation between features and proportion of discordant sleep

In [44]:
variables = ['chf', 'copd', 'CCI', 'hr_median', 'hr_q25', 'hr_q75', 'spo2_median', 'spo2_q25', 'spo2_q75', 'edw_bp_systolic_median', 'edw_bp_systolic_q25', 'edw_bp_systolic_q75', 
            'cpc_hfc_lfc_ratio_median', 'cpc_hfc_median', 'cpc_lfc_median', 'cpc_hfc_lfc_ratio_q25', 'cpc_hfc_lfc_ratio_q75',
             'hrv_nn_median', 'po2_arterial_mean', 'pco2_arterial_mean', 'hco3_arterial_mean', 'ph_arterial_mean', 'sofa_score_mean', 'sofa_score_max',
            'ibi_median', 'rr_10s_smooth_median', 'instability_30sec_median', 'ventilation_cvar_30sec_median', 'hrv_ulf_median', 'hrv_vlf_median', 'hrv_lf_median', 'hrv_hf_median', 
             'hrv_lfhf_median', 'hrv_rmssd_median', 'hrv_sd1sd2_median', 'opioids_sum', 'benzos_sum', 'antipsychotics_sum',
            ]
df_correlations = pd.DataFrame(columns=['pearson_c', 'pearson_p', 'spearman_c', 'spearman_p'], index=variables)

In [45]:
for variable in variables:
    
    var_values = Xy_icu[variable].replace('Series([], Freq: 100L, Name: hrv_nn, dtype: float32)', np.nan).astype(float).values
    discordant_prop = Xy_icu[discordant_proportion].values

    not_na = np.where(pd.notna(var_values))[0]
    var_values = var_values[not_na]
    discordant_prop = discordant_prop[not_na]

    pearson_c, pearson_p = pearsonr(var_values, discordant_prop)
    spearman_c, spearman_p = spearmanr(var_values, discordant_prop)
    
    df_correlations.loc[variable, :] = [pearson_c, pearson_p, spearman_c, spearman_p]

In [46]:
df_correlations['significance_pearson'] = df_correlations['pearson_p'] < 0.05
df_correlations['significance_spearman'] = df_correlations['spearman_p'] < 0.05

In [47]:
df_correlations.round(3)

,pearson_c,pearson_p,spearman_c,spearman_p,significance_pearson,significance_spearman
chf,-0.0484342,0.539236,-0.0261924,0.739976,False,False
copd,-0.149735,0.0564242,-0.151805,0.0530612,False,False
CCI,-0.0610756,0.438645,-0.0634193,0.421246,False,False
hr_median,-0.103006,0.190721,-0.07796,0.322579,False,False
hr_q25,-0.123631,0.115881,-0.0988157,0.209494,False,False
hr_q75,-0.126543,0.107478,-0.0742887,0.345958,False,False
spo2_median,-0.0507779,0.51976,-0.0391932,0.619383,False,False
spo2_q25,-0.0783765,0.319992,-0.0529948,0.501674,False,False
spo2_q75,-0.0417017,0.597122,-0.0349959,0.657408,False,False
edw_bp_systolic_median,-0.118746,0.165389,-0.111583,0.192592,False,False


In [48]:
savedir = '/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Results'

df_correlations.loc[['sofa_score_max', 'opioids_sum', 'benzos_sum', 'antipsychotics_sum']].to_csv(os.path.join(savedir, 'Discordant_Sleep_CorrelationsMedsSOFA.csv'))

### Linear Regression. 
Independent variable: proportion of discordance (transformed)  
Dependent variables: sleep-stagewise (both breathing and hrv models) features

In [49]:
# same variables as above for 'statistical difference check'
variables = ['hrv_nn',  'hrv_vlf', 'hrv_lf', 'hrv_hf', 'hrv_rmssd',
            'ibi', 'rr_10s_smooth', 'instability_30sec', 'ventilation_cvar_30sec',
            'cpc_lfc', 'cpc_hfc']

In [50]:
vars_dependent = []

for stage in ['N1', 'N2', 'N3', 'R']:
    for variable in variables:
        for modality in ['ecg_nn', 'amendsumeffort']:
            for statistic_extracted in ['mean', 'std', 'median', 'q25', 'q75']:
                vars_dependent += [variable + '_stagewise_agrelaxed_' + modality + '_' + stage + '_' + statistic_extracted]
                vars_dependent += [variable + '_stagewise_disrelaxed_' + modality + '_' + stage + '_' + statistic_extracted]
                
vars_dependent += ['hours_data_ecg_nn_all',
'hours_sleep_ecg_nn_all',
'stages_distribution_breathing_R',
'stages_distribution_breathing_N1',
'stages_distribution_breathing_N2',
'stages_distribution_breathing_N3']

In [51]:
X = Xy_icu[vars_dependent].copy()
X = X.replace('Series([], Name: hrv_nn, dtype: float32)', np.nan)
X = X.astype(float)
X = X.replace(np.inf, np.nan)
X.dropna(how='all', axis=1, inplace=True)
X.dropna(how='all', axis=0, inplace=True)
X = X.fillna(X.mean())
# X = X.values

In [52]:
X.shape

(163, 886)

In [53]:
list(df_fvalues.index[df_fvalues.Significance==True]);

In [59]:
y = Xy_icu[discordant_proportion].values
y = y + 0.0001
# transform y so it is more gaussian and unbounded (instead of [0, 1]):
y = np.log(np.divide(y, (1 - y)))

In [60]:
plt.figure(figsize=(4, 2))
plt.hist(y, histtype='step');

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [61]:
import statsmodels.api as sm

In [62]:
fvals = []
fpvals = []
nparams = []
rsquared = []
alpha_range = np.arange(0.01, 10, 0.1)
for alpha in alpha_range:
    lin_reg = sm.OLS(y, sm.add_constant(X)).fit_regularized(alpha=alpha, L1_wt=1)
    X_elasticnet = X.loc[:, lin_reg.params[1:] > 0]
    lin_reg = sm.OLS(y, sm.add_constant(X_elasticnet)).fit()
    fvals.append(lin_reg.fvalue)
    fpvals.append(lin_reg.f_pvalue)
    nparams.append(sum(lin_reg.params > 0))
    rsquared.append(lin_reg.rsquared)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/numpy/core/fromnumeric.py:2495: FutureWarning: Method .ptp is deprecated and will be removed in a future version. Use numpy.ptp instead.
  return ptp(axis=axis, out=out, **kwargs)


In [63]:
plt.close('all')

fig, ax = plt.subplots(4, 1, figsize=(6,4), sharex=True)
ax[0].plot(alpha_range, rsquared)
ax[0].set_ylabel('R squared')

ax[1].plot(alpha_range, fpvals)
ax[1].plot(alpha_range, [0.05] * len(alpha_range), c='r', ls='--', lw=0.8)
ax[1].set_yticks([0.05, 0.2, 0.4, 0.6])
ax[1].set_ylabel('F p-value')

ax[2].plot(alpha_range, fvals)
ax[2].set_ylabel('F statistic')
ax[2].set_yticks([0, 2, 4, 6, 8])
ax[3].plot(alpha_range, nparams)
ax[3].set_ylabel('# Variables\nincluded\nin Model')
ax[3].set_xlabel('Regularization strength LASSO (alpha)')

plt.tight_layout()
plt.subplots_adjust(hspace=0)
plt.savefig('discordance_linear_regression_results.tiff', dpi=500)
plt.savefig('discordance_linear_regression_results.svg', dpi=500)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [403]:
plt.subplots_adjust?

Signature:
plt.subplots_adjust(
    left=None,
    bottom=None,
    right=None,
    top=None,
    wspace=None,
    hspace=None,
)
Docstring:
Tune the subplot layout.

The parameter meanings (and suggested defaults) are::

  left = 0.125  # the left side of the subplots of the figure
  right = 0.9   # the right side of the subplots of the figure
  bottom = 0.1  # the bottom of the subplots of the figure
  top = 0.9     # the top of the subplots of the figure
  wspace = 0.2  # the amount of width reserved for space between subplots,
                # expressed as a fraction of the average axis width
  hspace = 0.2  # the amount of height reserved for space between subplots,
                # expressed as a fraction of the average axis height

The actual defaults are controlled by the rc file
File:      ~/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/matplotlib/pyplot.py
Type:      function


In [70]:
alpha = 4
lin_reg = sm.OLS(y, sm.add_constant(X)).fit_regularized(alpha=alpha, L1_wt=1)
X_elasticnet = X.loc[:, lin_reg.params[1:] > 0]
lin_reg = sm.OLS(y, sm.add_constant(X_elasticnet)).fit()
lin_reg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.424
Model:                            OLS   Adj. R-squared:                  0.143
Method:                 Least Squares   F-statistic:                     1.511
Date:                Fri, 23 Jul 2021   Prob (F-statistic):             0.0359
Time:                        10:36:42   Log-Likelihood:                -246.87
No. Observations:                 163   AIC:                             601.7
Df Residuals:                     109   BIC:                             768.8
Df Model:                          53                                         
Covariance Type:            nonrobust                                         
========================================================================================================================
                                                           coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------------
const                                                   -2.4044      0.753     -3.194      0.002      -3.896      -0.912
hrv_vlf_stagewise_agrelaxed_ecg_nn_N1_median         -6.601e-05      0.000     -0.234      0.815      -0.001       0.000
hrv_vlf_stagewise_disrelaxed_ecg_nn_N1_q25              -0.0003      0.000     -0.677      0.500      -0.001       0.001
hrv_vlf_stagewise_agrelaxed_amendsumeffort_N1_q75    -6.231e-05      0.000     -0.316      0.753      -0.000       0.000
hrv_lf_stagewise_disrelaxed_ecg_nn_N1_mean               0.0004      0.001      0.556      0.579      -0.001       0.002
hrv_lf_stagewise_disrelaxed_ecg_nn_N1_std             7.668e-05      0.000      0.163      0.871      -0.001       0.001
hrv_hf_stagewise_agrelaxed_ecg_nn_N1_std                -0.0004      0.000     -1.063      0.290      -0.001       0.000
hrv_hf_stagewise_agrelaxed_ecg_nn_N1_median           8.735e-05      0.001      0.132      0.895      -0.001       0.001
hrv_hf_stagewise_agrelaxed_ecg_nn_N1_q25                -0.0008      0.001     -1.386      0.169      -0.002       0.000
hrv_hf_stagewise_agrelaxed_ecg_nn_N1_q75                 0.0002      0.000      0.537      0.593      -0.000       0.001
hrv_hf_stagewise_agrelaxed_amendsumeffort_N1_mean       -0.0002      0.001     -0.360      0.720      -0.001       0.001
hrv_hf_stagewise_agrelaxed_amendsumeffort_N1_median      0.0003      0.001      0.396      0.693      -0.001       0.002
hrv_vlf_stagewise_disrelaxed_ecg_nn_N2_mean             -0.0011      0.001     -1.325      0.188      -0.003       0.001
hrv_vlf_stagewise_disrelaxed_ecg_nn_N2_std               0.0006      0.001      1.001      0.319      -0.001       0.002
hrv_vlf_stagewise_agrelaxed_ecg_nn_N2_q75            -3.252e-05      0.000     -0.112      0.911      -0.001       0.001
hrv_vlf_stagewise_disrelaxed_amendsumeffort_N2_std      -0.0001      0.000     -0.588      0.558      -0.001       0.000
hrv_vlf_stagewise_agrelaxed_amendsumeffort_N2_q75        0.0002      0.000      0.952      0.343      -0.000       0.001
hrv_lf_stagewise_disrelaxed_amendsumeffort_N2_q75       -0.0001      0.000     -0.247      0.806      -0.001       0.001
hrv_hf_stagewise_disrelaxed_ecg_nn_N2_mean               0.0002      0.000      0.926      0.356      -0.000       0.001
hrv_hf_stagewise_disrelaxed_amendsumeffort_N2_mean       0.0012      0.003      0.376      0.707      -0.005       0.007
hrv_hf_stagewise_disrelaxed_amendsumeffort_N2_std    -1.602e-05      0.001     -0.013      0.990      -0.003       0.002
hrv_hf_stagewise_disrelaxed_amendsumeffort_N2_median     0.0004      0.001      0.426      0.671      -0.001       0.002
hrv_hf_stagewise_disrelaxed_amendsumeffort_N2_q25       -0

In [400]:
lin_reg.params.to_csv('discordance_final_model_params.csv')

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: FutureWarning: The signature of `Series.to_csv` was aligned to that of `DataFrame.to_csv`, and argument 'header' will change its default value from False to True: please pass an explicit value to suppress this warning.
  """Entry point for launching an IPython kernel.


In [380]:
sum(lin_reg.params > 0)

32

In [381]:
print(X_elasticnet.shape)

(129, 56)


In [31]:
plt.figure(figsize=(4, 2))
plt.hist(Xy_icu[discordant_proportion], histtype='step')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(array([ 7.,  5., 11., 19., 13., 14., 20., 20., 10., 10.]),
 array([0.02580645, 0.11342346, 0.20104047, 0.28865748, 0.37627449,
        0.4638915 , 0.55150851, 0.63912553, 0.72674254, 0.81435955,
        0.90197656]),
 <a list of 1 Patch objects>)

In [52]:
plt.figure(figsize=(4, 2))
plt.hist(Xy_icu['sofa_score_max'], histtype='step')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(array([18., 32., 34., 12., 13.,  7.,  5.,  6.,  1.,  1.]),
 array([0. , 0.9, 1.8, 2.7, 3.6, 4.5, 5.4, 6.3, 7.2, 8.1, 9. ]),
 <a list of 1 Patch objects>)

In [58]:
variable = 'opioids_sum'
var_values = Xy_icu[variable].values
discordant_prop = Xy_icu[discordant_proportion].values
not_na = np.where(pd.notna(var_values))[0]
var_values = var_values[not_na]
discordant_prop = discordant_prop[not_na]

In [59]:
plt.figure()
plt.scatter(var_values, discordant_prop)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [330]:
variable1 = 'opioids_sum'
variable2 = 'stages_distribution_breathing_R'


var1_values = Xy_icu[variable1].astype(float).values
var2_values = Xy_icu[variable2].astype(float).values

not_na = np.where(pd.notna(var1_values) & pd.notna(var2_values))[0]
var1_values = var1_values[not_na]
var2_values = var2_values[not_na]

pearson_c, pearson_p = pearsonr(var1_values, var2_values)
spearman_c, spearman_p = spearmanr(var1_values, var2_values)
plt.figure(figsize=(3,3))
plt.scatter(var1_values, var2_values)
print(np.array([pearson_c, pearson_p, spearman_c, spearman_p]).round(4))


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:14: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

[0.0284 0.7536 0.0175 0.8465]


In [329]:
[x for x in Xy_icu.columns if 'R' in x]

['stages_distribution_breathing_R',
 'stages_distribution_breathing_agrelaxed_R',
 'stages_distribution_ecg_nn_R',
 'stages_distribution_ecg_nn_agrelaxed_R',
 'stages_distribution_expert_R',
 'hr_stagewise_agrelaxed_ecg_nn_R_mean',
 'hr_stagewise_agrelaxed_ecg_nn_R_std',
 'hr_stagewise_agrelaxed_ecg_nn_R_median',
 'hr_stagewise_agrelaxed_ecg_nn_R_q25',
 'hr_stagewise_agrelaxed_ecg_nn_R_q75',
 'hr_stagewise_agrelaxed_ecg_nn_R_sda',
 'hr_stagewise_agrelaxed_ecg_nn_R_asd',
 'hr_stagewise_agrelaxed_amendsumeffort_R_mean',
 'hr_stagewise_agrelaxed_amendsumeffort_R_std',
 'hr_stagewise_agrelaxed_amendsumeffort_R_median',
 'hr_stagewise_agrelaxed_amendsumeffort_R_q25',
 'hr_stagewise_agrelaxed_amendsumeffort_R_q75',
 'hr_stagewise_agrelaxed_amendsumeffort_R_sda',
 'hr_stagewise_agrelaxed_amendsumeffort_R_asd',
 'spo2_stagewise_agrelaxed_ecg_nn_R_mean',
 'spo2_stagewise_agrelaxed_ecg_nn_R_std',
 'spo2_stagewise_agrelaxed_ecg_nn_R_median',
 'spo2_stagewise_agrelaxed_ecg_nn_R_q25',
 'spo2_stagew

In [34]:
plt.figure()
plt.scatter(Xy_icu['opioids_sum'], Xy_icu['stages_distribution_breathing_R'] - Xy_icu['stages_distribution_breathing_agrelaxed_R'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [35]:
Xy_icu[['stages_distribution_breathing_R', 'stages_distribution_breathing_agrelaxed_R']]

,stages_distribution_breathing_R,stages_distribution_breathing_agrelaxed_R
5,15.161287,2.075401
11,4.635249,1.092474
17,0.000000,0.000000
83,2.428978,0.293151
86,3.171404,0.723282
89,6.882394,0.280770
95,49.467564,14.672682
104,8.472121,0.027268
107,50.090085,0.091113
113,0.000000,0.000000


In [238]:
Xy_icu[['stages_distribution_ecg_nn_all_R', 'stages_distribution_ecg_nn_agrelaxed_R']]

KeyError: "['stages_distribution_ecg_nn_all_R'] not in index"

In [ ]:
### opiates R investigation for breathing.

, , 

In [ ]:
for agreement in ['agreement_relaxed', 'agreement']:
    for modality in ['amendsumeffort', 'ecg_nn']:
        var_sleep = 'hours_sleep_MODALITY_all'.replace('MODALITY', modality) # all overlap sleep
        var_agreeing = 'hours_sleep_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        
        if agreement == 'agreement_relaxed':
             # all overlap sleep
            var_discordant = 'hours_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement) # discordant sleep among overlap sleep
        elif agreement == 'agreement':
            var_discordant = 'hours_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)

            
        summary_days_icu[var_discordant] = summary_days_icu[var_sleep] - summary_days_icu[var_agreeing]
        summary_days_sleeplab[var_discordant] = summary_days_sleeplab[var_sleep] - summary_days_sleeplab[var_agreeing]

### linear regression modelling, to check if we can predict on individual level if discordant sleep occurs.
currently: most important coefficients seem reasonable, but overall score is low. to be checked more carefully, i believe not needed for current paper.

In [68]:
# Xy_icu_imputed = Xy_icu.replace('Series([], Freq: 100L, Name: hrv_nn, dtype: float32)', np.nan).fillna(Xy_icu.mean())

import sklearn
from sklearn.linear_model import LinearRegression, ElasticNet, SGDClassifier
from sklearn.model_selection import GridSearchCV
# reg = LinearRegression().fit(Xy_icu_imputed[variables], Xy_icu_imputed[discordant_proportion])
# reg = ElasticNet(alpha=0.5, l1_ratio=0.25).fit(Xy_icu_imputed[variables], Xy_icu_imputed[discordant_proportion])

In [ ]:
parameters = {'alpha':[0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1], 'l1_ratio':[0.25, 0.5, 0.75]}

In [89]:
variables = ['sofa_score_mean',
             'opioids_sum',
#  'benzos_sum',
#  'antipsychotics_sum',
            ]

In [90]:
Xy_icu_imputed = Xy_icu[variables + [discordant_proportion]]
Xy_icu_imputed = Xy_icu_imputed.dropna(how='any')

In [91]:
Xy_icu_imputed.shape

(129, 3)

In [92]:
# discordant_proportion = 'stages_distribution_breathing_R'

In [93]:
x = Xy_icu_imputed[variables]
y = Xy_icu_imputed[discordant_proportion]

lin_reg = LinearRegression()
lin_reg = lin_reg.fit(x, y)

In [94]:
lin_reg.score(x, y)

0.03001131204679719

In [95]:
lin_reg.coef_

array([ 0.00456763, -0.00263014])

In [75]:
model = 'linear'

if model == 'linear':
    reg = ElasticNet()
    x = Xy_icu_imputed[variables]
    y = Xy_icu_imputed[discordant_proportion]
elif model == 'logistic':
    reg = SGDClassifier(loss="log", penalty="elasticnet")
    x = Xy_icu_imputed[variables]
    y = (Xy_icu_imputed[discordant_proportion] > 0.5).astype(int)
#     y = (Xy_icu_imputed[discordant_proportion] > 40).astype(int)
    

In [325]:
plt.figure()
plt.hist(Xy_icu_imputed[discordant_proportion])

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """Entry point for launching an IPython kernel.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(array([56., 14., 10.,  5., 17., 11.,  3.,  4.,  2.,  3.]),
 array([ 0.        ,  9.99999913, 19.99999826, 29.99999738, 39.99999651,
        49.99999564, 59.99999477, 69.9999939 , 79.99999302, 89.99999215,
        99.99999128]),
 <a list of 10 Patch objects>)

In [319]:
clf = GridSearchCV(reg, parameters, cv=10, scoring = 'roc_auc')

In [320]:
clf.fit(x, y)

GridSearchCV(cv=10, estimator=SGDClassifier(loss='log', penalty='elasticnet'),
             param_grid={'alpha': [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1],
                         'l1_ratio': [0.25, 0.5, 0.75]},
             scoring='roc_auc')

In [321]:
clf.cv_results_.keys()
pd.DataFrame(clf.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_l1_ratio,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002583,0.000174,0.002201,0.000101,0.01,0.25,"{'alpha': 0.01, 'l1_ratio': 0.25}",0.972222,0.777778,0.222222,0.611111,0.305556,0.343750,0.625,0.8125,0.68750,0.18750,0.554514,0.258507,1
1,0.002635,0.000132,0.002220,0.000116,0.01,0.5,"{'alpha': 0.01, 'l1_ratio': 0.5}",0.500000,0.777778,0.222222,0.611111,0.375000,0.562500,0.375,0.8125,0.62500,0.18750,0.504861,0.203107,2
2,0.002532,0.000311,0.002223,0.000307,0.01,0.75,"{'alpha': 0.01, 'l1_ratio': 0.75}",0.500000,0.500000,0.222222,0.555556,0.222222,0.687500,0.500,0.5000,0.71875,0.56250,0.496875,0.156277,9
3,0.002505,0.000120,0.002227,0.000255,0.02,0.25,"{'alpha': 0.02, 'l1_ratio': 0.25}",0.500000,0.777778,0.222222,0.611111,0.222222,0.562500,0.625,0.5000,0.62500,0.18750,0.483333,0.193355,11
4,0.002626,0.000242,0.002126,0.000100,0.02,0.5,"{'alpha': 0.02, 'l1_ratio': 0.5}",0.500000,0.500000,0.500000,0.500000,0.222222,0.343750,0.500,0.5000,0.62500,0.18750,0.437847,0.132750,17
5,0.002525,0.000122,0.002243,0.000140,0.02,0.75,"{'alpha': 0.02, 'l1_ratio': 0.75}",0.500000,0.500000,0.222222,0.555556,0.222222,0.468750,0.500,0.5000,0.62500,0.18750,0.428125,0.148326,18
6,0.002433,0.000161,0.002131,0.000093,0.05,0.25,"{'alpha': 0.05, 'l1_ratio': 0.25}",0.500000,0.500000,0.222222,0.555556,0.277778,0.468750,0.500,0.5000,0.62500,0.56250,0.471181,0.119093,12
7,0.002538,0.000386,0.002169,0.000117,0.05,0.5,"{'alpha': 0.05, 'l1_ratio': 0.5}",0.500000,0.500000,0.500000,0.500000,0.277778,0.421875,0.500,0.5000,0.50000,0.50000,0.469965,0.068166,13
8,0.002566,0.000123,0.002237,0.000087,0.05,0.75,"{'alpha': 0.05, 'l1_ratio': 0.75}",0.500000,0.500000,0.500000,0.500000,0.277778,0.375000,0.500,0.5000,0.50000,0.50000,0.465278,0.072768,14
9,0.002364,0.000125,0.002094,0.000064,0.1,0.25,"{'alpha': 0.1, 'l1_ratio': 0.25}",0.500000,0.555556,0.500000,0.500000,0.277778,0.250000,0.375,0.5000,0.50000,0.18750,0.414583,0.124616,19


In [322]:
clf.cv_results_['mean_test_score']

array([0.55451389, 0.50486111, 0.496875  , 0.48333333, 0.43784722,
       0.428125  , 0.47118056, 0.46996528, 0.46527778, 0.41458333,
       0.39496528, 0.5       , 0.44375   , 0.5       , 0.5       ,
       0.40711806, 0.5       , 0.4875    , 0.44496528, 0.503125  ,
       0.5       ])

In [253]:
clf.cv_results_['params']

[{'alpha': 0.01, 'l1_ratio': 0.25},
 {'alpha': 0.01, 'l1_ratio': 0.5},
 {'alpha': 0.01, 'l1_ratio': 0.75},
 {'alpha': 0.02, 'l1_ratio': 0.25},
 {'alpha': 0.02, 'l1_ratio': 0.5},
 {'alpha': 0.02, 'l1_ratio': 0.75},
 {'alpha': 0.05, 'l1_ratio': 0.25},
 {'alpha': 0.05, 'l1_ratio': 0.5},
 {'alpha': 0.05, 'l1_ratio': 0.75},
 {'alpha': 0.1, 'l1_ratio': 0.25},
 {'alpha': 0.1, 'l1_ratio': 0.5},
 {'alpha': 0.1, 'l1_ratio': 0.75},
 {'alpha': 0.2, 'l1_ratio': 0.25},
 {'alpha': 0.2, 'l1_ratio': 0.5},
 {'alpha': 0.2, 'l1_ratio': 0.75},
 {'alpha': 0.5, 'l1_ratio': 0.25},
 {'alpha': 0.5, 'l1_ratio': 0.5},
 {'alpha': 0.5, 'l1_ratio': 0.75},
 {'alpha': 1, 'l1_ratio': 0.25},
 {'alpha': 1, 'l1_ratio': 0.5},
 {'alpha': 1, 'l1_ratio': 0.75}]

In [219]:
print(reg.score(Xy_icu_imputed[variables], Xy_icu_imputed[discordant_proportion]))

0.10608757890812248


In [220]:
reg.coef_

array([ 0.00000000e+00, -0.00000000e+00,  0.00000000e+00, -0.00000000e+00,
        0.00000000e+00, -1.76249985e-03, -0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -0.00000000e+00, -1.87418367e-03,
        0.00000000e+00, -0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00, -2.68387813e-03,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00,  4.08875601e-03, -0.00000000e+00, -0.00000000e+00,
        1.86704952e-04, -1.08553102e-05, -6.32607122e-05,  2.21462782e-05,
       -0.00000000e+00, -1.30367172e-04,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00, -0.00000000e+00])

In [221]:
df_correlations.shape

(38, 8)

In [222]:
reg.coef_.shape

(38,)

In [223]:
df_correlations['LinearRegCoef'] = reg.coef_

In [224]:
df_correlations['LinearRegCoefAbs'] = df_correlations['LinearRegCoef'].abs()

In [225]:
df_correlations.sort_values(by='LinearRegCoefAbs', ascending=False)[:10]

,pearson_c,pearson_p,spearman_c,spearman_p,significance_pearson,significance_spearman,LinearRegCoef,LinearRegCoefAbs
rr_10s_smooth_median,0.206869,0.00933511,0.207834,0.00900347,True,True,0.004089,0.004089
po2_arterial_mean,-0.429372,0.0665678,-0.4,0.0897187,False,False,-0.002684,0.002684
edw_bp_systolic_q75,-0.195113,0.0223232,-0.227258,0.00757041,True,True,-0.001874,0.001874
hr_q75,-0.12906,0.107193,-0.0988418,0.218097,False,False,-0.001762,0.001762
hrv_ulf_median,0.0775241,0.344077,-0.0492297,0.548318,False,False,0.000187,0.000187
hrv_rmssd_median,0.0266267,0.745534,0.0504723,0.538256,False,False,-0.000130,0.000130
hrv_lf_median,0.00992397,0.903741,-0.028008,0.732821,False,False,-0.000063,0.000063
hrv_hf_median,0.0488961,0.551036,0.0321245,0.695374,False,False,0.000022,0.000022
hrv_vlf_median,0.0211977,0.79614,-0.0806519,0.324901,False,False,-0.000011,0.000011
ventilation_cvar_30sec_median,-0.306983,9.20672e-05,-0.323444,3.59642e-05,True,True,-0.000000,0.000000
